In [1]:
from datasets import load_dataset
from pathlib import Path
import pickle
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

data_dir = Path("./data")
model_dir = Path("./.cache/models")
result_dir = Path("./results/10000_samples_HaluEval_Qwen2.5-7B-Instruct_v1")

In [2]:
dataset = load_dataset('pminervini/HaluEval', 'qa_samples', cache_dir=str(data_dir))
dataset = dataset['data']

In [3]:
model_name = "Qwen/Qwen2.5-7B-Instruct"
model = AutoModelForCausalLM.from_pretrained(model_name, cache_dir=model_dir, device_map="auto", trust_remote_code=True, torch_dtype=torch.float16)
tokenizer = AutoTokenizer.from_pretrained(model_name, cache_dir=model_dir, trust_remote_code=True)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

In [4]:
def get_tokenized_ids(prompt, answer):
    text = f"{prompt.strip()} {answer.strip()}"
    otext = prompt.strip()
    id1 = tokenizer(text.strip(), return_tensors='pt')['input_ids'].tolist()[0]
    id2 = tokenizer(otext.strip(), return_tensors='pt')['input_ids'].tolist()[0]
    start_at = -1
    for i in range(len(id1)):
        if i >= len(id2) or id1[i] != id2[i]:
            start_at = i
            break
    return [id1], start_at

In [5]:
generated_embeddings = []
labels = []
for i in tqdm(range(0, len(dataset))):
    context = dataset['knowledge'][i]
    question = dataset['question'][i]
    answer = dataset['answer'][i]
    hallu = dataset['hallucination'][i]
    user_prompt = """You are a precise analytical system. Please answer the following question based on the provided knowledge background. You must answer the question as directly and accurately as possible, only return your final answer without any additional information or explanation or your resoning process.
Knowledge background: <knowledge background> {context} </knowledge background>
Question: <question> {question} </question>
Your answer:"""
    # tok_in_prompt = tokenizer(user_prompt, return_tensors="pt", add_special_tokens=True).input_ids
    # prompt_len = tok_in_prompt.shape[1]
    # full_text = user_prompt + answer
    # inputs = tokenizer(full_text, return_tensors="pt")
    ids, start_at = get_tokenized_ids(user_prompt, answer)
    with torch.no_grad():
        outputs = model(torch.tensor(ids).to(model.device), output_hidden_states=True)
    last_token_embeddings = outputs.hidden_states[-1][0, start_at-1:, :].to(torch.float32).detach().cpu()
    generated_embeddings.append(last_token_embeddings)
    if hallu.lower() == "yes":
        labels.append(1)
    else:
        labels.append(0)

with open(result_dir / "contextual_embs.pkl", "wb") as pkl_file:
    pickle.dump([generated_embeddings, labels], pkl_file)

100%|██████████| 10000/10000 [12:16<00:00, 13.58it/s]


In [6]:
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from model import *

gpu = "0"
device = torch.device(f"cuda:{gpu}" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

def load_contextual_embeddings_by_id(embeddings_path):
    with open(embeddings_path, "rb") as f:
        generated_embeddings, labels = pickle.load(f)
    pooled_embeddings = []
    for i in range(len(generated_embeddings)):
        embeddings = generated_embeddings[i]
        emb_sequence = torch.tensor(embeddings, dtype=torch.float32)
        print(f"emb_sequence shape: {emb_sequence.shape}")
        # pooled_emb = torch.mean(emb_sequence, dim=0)
        pooled_emb = torch.max(emb_sequence, dim=0).values
        pooled_embeddings.append(pooled_emb)
    pooled_embeddings = torch.stack(pooled_embeddings)
    return pooled_embeddings, labels


def gen_classifier_roc(inputs, labels, model, save_name="classifier_model.pt"):
    X_train, X_test, y_train, y_test = train_test_split(inputs, labels, test_size=0.2, random_state=123)
    classifier_model = model(X_train.shape[1]).to(device)
    X_train = torch.tensor(X_train).to(device)
    y_train = torch.tensor(y_train).to(torch.long).to(device)
    X_test = torch.tensor(X_test).to(device)
    y_test = torch.tensor(y_test).to(torch.long).to(device)

    optimizer = torch.optim.AdamW(classifier_model.parameters(), lr=learning_rate, weight_decay=weight_decay)

    for _ in tqdm(range(1001)):
        optimizer.zero_grad()
        sample = torch.randperm(X_train.shape[0])[:batch_size]
        pred = classifier_model(X_train[sample])
        loss = torch.nn.functional.cross_entropy(pred, y_train[sample])
        loss.backward()
        optimizer.step()
    # torch.save(classifier_model.state_dict(), f"{save_model_dir}/{save_name}")

    classifier_model.eval()
    with torch.no_grad():
        pred = torch.nn.functional.softmax(classifier_model(X_test), dim=1)
        prediction_classes = (pred[:, 1] > 0.5).type(torch.long).cpu()
        return roc_auc_score(y_test.cpu(), pred[:, 1].cpu()), (prediction_classes.numpy() == y_test.cpu().numpy()).mean()

Using device: cuda:0


In [7]:
embeddings_path = result_dir / "contextual_embs.pkl"
batch_size = 128
learning_rate = 1e-4
weight_decay = 1e-2

classifier = SingleMLP_Classifier

inputs, labels = load_contextual_embeddings_by_id(embeddings_path)
print(f"Loaded {len(inputs)} contextual embeddings and {len(labels)} labels")

context_emb_roc, context_emb_acc = gen_classifier_roc(inputs, labels, model=classifier, save_name="mlp_context_emb.pt")
print(f"context_emb_roc: {context_emb_roc}")
print(f"context_emb_acc: {context_emb_acc}")

with open(result_dir / "score_result.txt", "a", encoding="utf-8") as text_file:
    text_file.write(f"context_emb_roc {context_emb_roc}\n")
    text_file.write(f"context_emb_acc {context_emb_acc}\n")

/tmp/ipykernel_3991922/226809130.py:15: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  emb_sequence = torch.tensor(embeddings, dtype=torch.float32)


emb_sequence shape: torch.Size([8, 3584])
emb_sequence shape: torch.Size([2, 3584])
emb_sequence shape: torch.Size([4, 3584])
emb_sequence shape: torch.Size([2, 3584])
emb_sequence shape: torch.Size([7, 3584])
emb_sequence shape: torch.Size([10, 3584])
emb_sequence shape: torch.Size([17, 3584])
emb_sequence shape: torch.Size([4, 3584])
emb_sequence shape: torch.Size([6, 3584])
emb_sequence shape: torch.Size([9, 3584])
emb_sequence shape: torch.Size([4, 3584])
emb_sequence shape: torch.Size([23, 3584])
emb_sequence shape: torch.Size([3, 3584])
emb_sequence shape: torch.Size([6, 3584])
emb_sequence shape: torch.Size([22, 3584])
emb_sequence shape: torch.Size([16, 3584])
emb_sequence shape: torch.Size([23, 3584])
emb_sequence shape: torch.Size([23, 3584])
emb_sequence shape: torch.Size([12, 3584])
emb_sequence shape: torch.Size([2, 3584])
emb_sequence shape: torch.Size([16, 3584])
emb_sequence shape: torch.Size([11, 3584])
emb_sequence shape: torch.Size([5, 3584])
emb_sequence shape: torc

/tmp/ipykernel_3991922/226809130.py:27: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  X_train = torch.tensor(X_train).to(device)
/tmp/ipykernel_3991922/226809130.py:29: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  X_test = torch.tensor(X_test).to(device)
100%|██████████| 1001/1001 [00:04<00:00, 227.87it/s]

context_emb_roc: 0.975073975073975
context_emb_acc: 0.962
